In [ ]:
# ============================================================
#  - Uses pt_cache (.pt) voxels
#  - Metric inputs scaled by METRIC_SCALE
#  - Metric reconstruction loss weighted by 10x
# ============================================================

import os
import sys
import time
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import torch.backends.cudnn as cudnn
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

WORKING_DIR = Path.cwd()
if (WORKING_DIR / "code" / "00_training").is_dir():
    PROJECT_ROOT = WORKING_DIR
elif WORKING_DIR.name == "00_training" and WORKING_DIR.parent.name == "code":
    PROJECT_ROOT = WORKING_DIR.parents[1]
else:
    raise RuntimeError("Run this notebook from the repository root or code/00_training.")
TRAINING_DIR = PROJECT_ROOT / "code" / "00_training"
sys.path.insert(0, str(TRAINING_DIR))

import training_config
from multimodal_autoencoder import MultiModalAutoencoder   # latent=48

# ------------------------------------------------------------
# Global settings
# ------------------------------------------------------------
cudnn.benchmark = True
scaler = GradScaler()

opt = training_config.args_parser()
METRIC_SCALE = 10.0
print("METRIC_SCALE (train):", METRIC_SCALE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Batch size (opt.batch_size):", opt.batch_size)

FIXED_P16 = 0.95
FIXED_P32 = 0.95
FIXED_P64 = 0.95

# ------------------------------------------------------------
# Paths and CSV
# ------------------------------------------------------------
voxel_dir   = (PROJECT_ROOT / "../../../GRA/3D_Voxel/Total_Dataset").as_posix()
data_path   = (PROJECT_ROOT / "data/01_supplier_identification/main_split_70_30").as_posix()
result_path = (TRAINING_DIR / "checkpoints").as_posix()

csv_path      = os.path.join(data_path, "train_dataset_without_quantity.csv")
loss_csv_path = os.path.join(result_path, "training_loss.csv")
ckpt_path     = os.path.join(result_path, "multimodal_autoencoder_epoch_000.pth")

os.makedirs(result_path, exist_ok=True)

if not os.path.exists(csv_path):
    files = [f for f in os.listdir(voxel_dir) if f.endswith(".binvox")]
    pd.DataFrame(files, columns=["FileName"]).to_csv(csv_path, index=False)

df = pd.read_csv(csv_path)

if "filename" not in df.columns:
    if "FileName" in df.columns:
        df.rename(columns={"FileName": "filename"}, inplace=True)
    else:
        raise KeyError("CSV must contain 'filename' or 'FileName'.")

existing = set(os.listdir(voxel_dir))
df = df[df["filename"].isin(existing)].reset_index(drop=True)

col_map = {
    "LogScaled_Time": "TimeNorm",
    "LogScaled_Cost": "CostNorm",
    "LogScaled_Quantity": "QtyNorm",
    "LogScaled_Tolerance": "TolNorm",
    "LogScaled_Density": "DensityNorm",
    "LogScaled_Service_Temperature": "ServiceTempNorm",
    "LogScaled_Ultimate_Tensile": "UTSNrm",
}
for std, alias in col_map.items():
    if std not in df.columns and alias in df.columns:
        df[std] = df[alias]

required_cols = [
    "LogScaled_Time",
    "LogScaled_Cost",
    "LogScaled_Quantity",
    "LogScaled_Tolerance",
    "LogScaled_Density",
    "LogScaled_Service_Temperature",
    "LogScaled_Ultimate_Tensile",
]
use_meta = all(c in df.columns for c in required_cols)
if not use_meta:
    print("[WARN] Metadata columns are missing. Time/Cost/Quantity/Tolerance/Materials will be set to 0.")

df.to_csv(csv_path, index=False)

print("Number of samples (CSV):", len(df))
print("use_meta:", use_meta)

# ------------------------------------------------------------
# Loss utilities
# ------------------------------------------------------------
def dice_loss_with_logits(logits: torch.Tensor,
                          target: torch.Tensor,
                          eps: float = 1e-6) -> torch.Tensor:
    """
    logits: [B,1,D,H,W], target: [B,1,D,H,W] in {0,1}
    """
    prob = torch.sigmoid(logits)
    dims = [1, 2, 3, 4]
    num = 2.0 * (prob * target).sum(dim=dims)
    den = (prob * prob).sum(dim=dims) + (target * target).sum(dim=dims) + eps
    dice = num / den
    return 1.0 - dice.mean()


def make_bce_with_pos_weight_from_batch(target: torch.Tensor,
                                        min_pw: float = 1.0,
                                        max_pw: float = 50.0):
    """
    """
    r = target.float().mean().item()            # positive ratio
    pos_w = (1.0 - r) / max(r, 1e-6)
    pos_w = float(max(min_pw, min(pos_w, max_pw)))
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_w], device=target.device)
    )
    return loss_fn, r, pos_w

# ------------------------------------------------------------
# ------------------------------------------------------------
class VoxelFromPtDataset(Dataset):
    """
    """
    def __init__(self, df: pd.DataFrame, voxel_dir: str):
        super().__init__()
        self.voxel_dir = voxel_dir
        self.pt_dir = os.path.join(voxel_dir, "pt_cache")

        if "filename" not in df.columns:
            if "FileName" in df.columns:
                df = df.rename(columns={"FileName": "filename"})
            else:
                raise KeyError("CSV must contain 'filename' or 'FileName'.")

        all_pt_files = [f for f in os.listdir(self.pt_dir) if f.endswith(".pt")]
        pt_set = set(all_pt_files)

        voxel_files = []
        valid_rows = []
        missing_count = 0

        for idx, row in df.iterrows():
            fname = row["filename"]
            pt_name = fname.replace(".binvox", ".pt")
            if pt_name in pt_set:
                voxel_files.append(os.path.join(self.pt_dir, pt_name))
                valid_rows.append(row)
            else:
                missing_count += 1
                if missing_count <= 1:
                    print("[WARN] No matching file in pt_cache. Example:", pt_name)

            if (idx + 1) % 1000 == 0:
                pass

        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)
        self.voxel_files = voxel_files


    def __len__(self):
        return len(self.voxel_files)

    def __getitem__(self, idx):
        # -----------------------------
        # -----------------------------
        voxel = torch.load(self.voxel_files[idx])

        # case 1: [D, H, W]
        if voxel.ndim == 3:
            voxel = voxel.unsqueeze(0)            # [1, D, H, W]

        elif voxel.ndim == 4:
            pass

        elif voxel.ndim == 5:
            if voxel.size(0) == 1:
                voxel = voxel.squeeze(0)          # [1, D, H, W]
            else:
                raise RuntimeError(f"Unexpected 5D voxel shape: {voxel.shape}")
        else:
            raise RuntimeError(f"Unexpected voxel ndim: {voxel.ndim}, shape={voxel.shape}")

        # -----------------------------
        # -----------------------------
        if use_meta:
            row = self.df.iloc[idx]
            t_cap   = torch.tensor(row["LogScaled_Time"],      dtype=torch.float32)
            c_cap   = torch.tensor(row["LogScaled_Cost"],      dtype=torch.float32)
            q_cap   = torch.tensor(row["LogScaled_Quantity"],  dtype=torch.float32)
            tol_cap = torch.tensor(row["LogScaled_Tolerance"], dtype=torch.float32)
            m_cap   = torch.tensor(
                [
                    row["LogScaled_Density"],
                    row["LogScaled_Service_Temperature"],
                    row["LogScaled_Ultimate_Tensile"],
                ],
                dtype=torch.float32,
            )  # [3]
        else:
            t_cap   = torch.tensor(0.0,          dtype=torch.float32)
            c_cap   = torch.tensor(0.0,          dtype=torch.float32)
            q_cap   = torch.tensor(0.0,          dtype=torch.float32)
            tol_cap = torch.tensor(0.0,          dtype=torch.float32)
            m_cap   = torch.zeros(3, dtype=torch.float32)

        # voxel: [C,D,H,W], t/c/q/tol: scalar, m_cap: [3]
        return voxel, t_cap, c_cap, q_cap, tol_cap, m_cap

# ------------------------------------------------------------
# Dataset, DataLoader, model, optimizer
# ------------------------------------------------------------
train_dataset = VoxelFromPtDataset(df=df, voxel_dir=voxel_dir)

batch_size = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=False,
)

print("[DEBUG] DataLoader sanity check:")
first_batch = next(iter(train_loader))
voxel_b, t_b, c_b, q_b, tol_b, m_b = first_batch
print("voxel_batch.shape =", voxel_b.shape)
print("t_batch.shape     =", t_b.shape)
print("m_batch.shape     =", m_b.shape)

model = MultiModalAutoencoder(normalize_shape=True).to(device)
optimizer = optim.Adam(model.parameters(), lr=opt.lr, betas=opt.betas)
mse = nn.MSELoss()

writer = SummaryWriter(log_dir=os.path.join(result_path, "logs"))

last_epoch = 0
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    try:
        last_epoch = int(os.path.basename(ckpt_path).split("_")[-1].split(".")[0])
        print(f"Resuming from epoch {last_epoch}")
    except Exception:
        print("Checkpoint loaded. Failed to parse epoch index.")
else:
    print("No checkpoint found, training from scratch.")

# ------------------------------------------------------------
# Training hyperparameters
# ------------------------------------------------------------
num_epochs = opt.epochs
save_freq  = 5

lambda_vox_bce  = 10.0
lambda_vox_dice = 10.0

lambda_t   = 1.0
lambda_c   = 1.0
lambda_tol = 1.0
lambda_m   = 1.0

has_levelwise = False
try:
    model.shape_dec.set_skip_drop_ps(FIXED_P16, FIXED_P32, FIXED_P64)
    has_levelwise = True
except Exception:
    try:
        avg_p = float((FIXED_P16 + FIXED_P32 + FIXED_P64) / 3.0)
        model.shape_dec.set_skip_drop_p(avg_p)
    except Exception:
        pass

# ------------------------------------------------------------
# Training loop (metric loss ×10, batch_size>1)
# ------------------------------------------------------------
for epoch in range(last_epoch, last_epoch + num_epochs):
    t0 = time.time()
    model.train()

    if has_levelwise:
        try:
            model.shape_dec.set_skip_drop_ps(FIXED_P16, FIXED_P32, FIXED_P64)
        except Exception:
            pass
    else:
        try:
            avg_p = float((FIXED_P16 + FIXED_P32 + FIXED_P64) / 3.0)
            model.shape_dec.set_skip_drop_p(avg_p)
        except Exception:
            pass

    steps = 0
    ep_total = ep_vox = ep_metrics = 0.0
    sum_bce = sum_dice = 0.0
    sum_t = sum_c = sum_tol = sum_m = 0.0

    with tqdm(train_loader,
              desc=f"Epoch [{epoch+1}/{last_epoch+num_epochs}]",
              unit="batch") as pbar:

        for batch_idx, batch in enumerate(pbar):
            voxel, t_cap, c_cap, q_cap, tol_cap, m_cap = batch
            # voxel: [B,1,D,H,W], t/c/q/tol: [B], m_cap: [B,3]

            voxel   = voxel.to(device, non_blocking=True).float()
            t_cap   = t_cap.to(device, non_blocking=True).float()        # [B]
            c_cap   = c_cap.to(device, non_blocking=True).float()        # [B]
            q_cap   = q_cap.to(device, non_blocking=True).float()
            tol_cap = tol_cap.to(device, non_blocking=True).float()      # [B]
            m_cap   = m_cap.to(device, non_blocking=True).float()        # [B,3]

            t_in   = t_cap   * METRIC_SCALE
            c_in   = c_cap   * METRIC_SCALE
            q_in   = torch.zeros_like(t_in)
            tol_in = tol_cap * METRIC_SCALE
            m_in   = m_cap   * METRIC_SCALE

            optimizer.zero_grad(set_to_none=True)

            # AMP forward
            with autocast():
                x_logits, (t_hat, c_hat, q_hat, tol_hat, m_hat), _ = model(
                    voxel, t_in, c_in, q_in, tol_in, m_in
                )

                # Shape loss: BCE + Dice
                bce_obj, pos_ratio, pos_weight = make_bce_with_pos_weight_from_batch(
                    voxel, min_pw=1.0, max_pw=50.0
                )
                loss_bce  = bce_obj(x_logits, voxel)
                loss_dice = dice_loss_with_logits(x_logits, voxel)
                loss_vox  = lambda_vox_bce * loss_bce + lambda_vox_dice * loss_dice

                loss_t   = mse(t_hat.view_as(t_cap),     t_cap)
                loss_c   = mse(c_hat.view_as(c_cap),     c_cap)
                loss_tol = mse(tol_hat.view_as(tol_cap), tol_cap)
                loss_m   = mse(m_hat,                    m_cap)   # [B,3] vs [B,3]

                loss_metrics = 10.0 * (
                    lambda_t   * loss_t +
                    lambda_c   * loss_c +
                    lambda_tol * loss_tol +
                    lambda_m   * loss_m
                )

                loss = loss_vox + loss_metrics

            # backward (AMP)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            scaler.step(optimizer)
            scaler.update()

            steps     += 1
            ep_total  += float(loss.item())
            ep_vox    += float(loss_vox.item())
            ep_metrics+= float(loss_metrics.item())
            sum_bce   += float(loss_bce.item())
            sum_dice  += float(loss_dice.item())
            sum_t     += float(loss_t.item())
            sum_c     += float(loss_c.item())
            sum_tol   += float(loss_tol.item())
            sum_m     += float(loss_m.item())

            if (batch_idx % 100 == 0) or (batch_idx == len(train_loader) - 1):
                vox_over_metrics = float(loss_vox.item() / (loss_metrics.item() + 1e-8))
                bce_over_dice    = float((loss_bce.item() + 1e-12) / (loss_dice.item() + 1e-12))
                pbar.set_postfix(
                    Loss=f"{loss.item():.3f}",
                    Vox=f"{loss_vox.item():.3f}",
                    Mx=f"{loss_metrics.item():.3f}",
                    rVM=f"{vox_over_metrics:.2f}",
                    rBD=f"{bce_over_dice:.2f}",
                    P16=f"{FIXED_P16:.2f}",
                    P32=f"{FIXED_P32:.2f}",
                    P64=f"{FIXED_P64:.2f}",
                )

    def mean(v: float) -> float:
        return v / max(steps, 1)

    mean_total, mean_vox, mean_metrics = map(mean, (ep_total, ep_vox, ep_metrics))
    mean_bce, mean_dice = map(mean, (sum_bce, sum_dice))
    mean_t, mean_c, mean_tol, mean_m = map(
        mean, (sum_t, sum_c, sum_tol, sum_m)
    )

    print(
        f"Epoch [{epoch+1}/{last_epoch+num_epochs}] "
        f"Total {mean_total:.4f} | Vox {mean_vox:.4f} (BCE {mean_bce:.4f}, Dice {mean_dice:.4f}) "
        f"| Metrics {mean_metrics:.4f} "
        f"| Time {time.time()-t0:.1f}s"
    )

    global_step = epoch
    writer.add_scalar("Loss/Total",   mean_total,  global_step)
    writer.add_scalar("Loss/Vox",     mean_vox,    global_step)
    writer.add_scalar("Loss/Metrics", mean_metrics,global_step)
    writer.add_scalar("Loss/Vox_BCE", mean_bce,    global_step)
    writer.add_scalar("Loss/Vox_Dice",mean_dice,   global_step)
    writer.add_scalar("Loss/T",       mean_t,      global_step)
    writer.add_scalar("Loss/C",       mean_c,      global_step)
    writer.add_scalar("Loss/Tol",     mean_tol,    global_step)
    writer.add_scalar("Loss/M",       mean_m,      global_step)

    if (epoch + 1) % save_freq == 0:
        ckpt_out = os.path.join(result_path, f"multimodal_autoencoder_epoch_{epoch+1:03d}.pth")
        torch.save(model.state_dict(), ckpt_out)
        print("Saved checkpoint to:", ckpt_out)
